##  Establish MySQL Database Connection

Connect Python to the MySQL database `telco_churn_db` using `mysql.connector`.

The connection will be used to create the churn table, insert the cleaned dataset, and execute SQL queries for customer churn analysis.

In [ ]:
import pandas as pd
import mysql.connector

# Load cleaned data
df_clean = pd.read_csv('../Dataset/telco_churn_cleaned.csv')
print(df_clean.shape)

# Connect to MySQL
conn = mysql.connector.connect(
    host='localhost',
    user='root',
    password='your_password',
    database='telco_churn_db'
)
cursor = conn.cursor()

# Drop table if it already exists (so re-running this cell is safe)
cursor.execute("DROP TABLE IF EXISTS telco_churn;")

# Build CREATE TABLE statement dynamically based on df_clean's columns/dtypes
def map_dtype(dtype):
    if pd.api.types.is_integer_dtype(dtype):
        return "INT"
    elif pd.api.types.is_float_dtype(dtype):
        return "FLOAT"
    else:
        return "VARCHAR(255)"

columns_sql = ", ".join([f"`{col}` {map_dtype(dtype)}" for col, dtype in df_clean.dtypes.items()])
create_table_query = f"CREATE TABLE telco_churn ({columns_sql});"
cursor.execute(create_table_query)

# Prepare INSERT statement
cols = ", ".join([f"`{col}`" for col in df_clean.columns])
placeholders = ", ".join(["%s"] * len(df_clean.columns))
insert_query = f"INSERT INTO telco_churn ({cols}) VALUES ({placeholders})"

# Convert DataFrame rows to list of tuples, replacing NaN with None
data = df_clean.where(pd.notnull(df_clean), None).values.tolist()

# Bulk insert
cursor.executemany(insert_query, data)
conn.commit()

print(f"Inserted {cursor.rowcount} rows successfully!")

cursor.close()
conn.close()

(7043, 30)
Inserted 7043 rows successfully!


In [2]:
import pandas as pd
import mysql.connector

conn = mysql.connector.connect(
    host='localhost',
    user='root',
    password='Himanshu@123',
    database='telco_churn_db'
)

cursor = conn.cursor()

## 1. Monthly Revenue Associated with Churned Customers
Calculate how much Monthly Recurring Revenue (MRR) is being lost to churned 
customers, and what percentage of total revenue that represents.

In [22]:
query1 = """
SELECT
    SUM(CASE
        WHEN `Churn Label` = 'Yes'
        THEN `Monthly Charges`
        ELSE 0
    END) AS churned_monthly_revenue,

    SUM(`Monthly Charges`) AS total_monthly_revenue,

    ROUND(
        SUM(CASE
            WHEN `Churn Label` = 'Yes'
            THEN `Monthly Charges`
            ELSE 0
        END)
        / SUM(`Monthly Charges`) * 100,
        2
    ) AS revenue_at_risk_pct

FROM telco_churn;
"""

cursor.execute(query1)

result = cursor.fetchall()
df_query1 = pd.DataFrame(result, columns=["churned_monthly_revenue", "total_monthly_revenue", "revenue_at_risk_pct"])
df_query1

,churned_monthly_revenue,total_monthly_revenue,revenue_at_risk_pct
0,139130.850014,456116.600145,30.5


### Observation

- Churned customers are associated with approximately **139,130.85** in monthly charges.
- Total monthly charges across the customer base are approximately **456,116.60**.
- The monthly charges associated with churned customers represent approximately **30.50%** of total monthly charges in the dataset.
- This indicates that churn is associated with a substantial amount of recurring monthly revenue exposure.

## 2. High-Value Customers at Risk
Identify currently active customers with both high CLTV and a high Churn Score — 
valuable customers the company is at risk of losing.

In [21]:
query2 = """
SELECT
    `CustomerID`,
    `CLTV`,
    `Churn Score`,
    `Monthly Charges`,
    `Tenure Months`,
    `Contract`,
    `Internet Service`
FROM telco_churn
WHERE `Churn Label` = 'No'
  AND `CLTV` > (SELECT AVG(`CLTV`) FROM telco_churn)
  AND `Churn Score` > (SELECT AVG(`Churn Score`) FROM telco_churn)
ORDER BY `Churn Score` DESC, `CLTV` DESC;
"""

cursor.execute(query2)

result = cursor.fetchall()

df_query2 = pd.DataFrame(
    result,
    columns=[
        "CustomerID",
        "CLTV",
        "Churn Score",
        "Monthly Charges",
        "Tenure Months",
        "Contract",
        "Internet Service"
    ]
)

df_query2

,CustomerID,CLTV,Churn Score,Monthly Charges,Tenure Months,Contract,Internet Service
0,5914-XRFQB,6479,80,115.80,72,Two year,Fiber optic
1,5074-FBGHB,6443,80,104.65,64,One year,Fiber optic
2,0619-OLYUR,6437,80,111.90,72,Two year,Fiber optic
3,7375-WMVMT,6412,80,95.50,71,Two year,Fiber optic
4,5125-CNDSP,6406,80,40.60,63,One year,DSL
...,...,...,...,...,...,...,...
1056,0536-ESJEP,4544,59,74.55,54,Two year,DSL
1057,2480-JZOSN,4517,59,20.65,1,Month-to-month,No
1058,7921-BEPCI,4515,59,59.75,72,Two year,DSL
1059,3865-ZYKAD,4503,59,84.90,69,Month-to-month,Fiber optic


### Observation

- The query identifies **1,061 active customers** whose CLTV and Churn Score are both above their respective dataset averages.
- These customers represent a group with relatively high customer lifetime value combined with elevated churn risk according to the existing Churn Score.
- The results can be used to identify customer records for further retention analysis.
- The Churn Score used here is an existing score in the IBM dataset and is not the project's own machine-learning prediction.

## 3. Revenue-Weighted Segment Ranking
Rank Contract × Internet Service combinations by total revenue lost to churn, 
not just churn rate — a segment can have moderate churn % but still cause the 
biggest Churned Monthly Revenue if it's large or expensive.


In [5]:
query3 = """
SELECT
    `Contract`,
    `Internet Service`,
    
    COUNT(*) AS total_customers,
    
    SUM(
        CASE
            WHEN `Churn Label` = 'Yes'
            THEN 1
            ELSE 0
        END
    ) AS churned_customers,
    
    SUM(
        CASE
            WHEN `Churn Label` = 'Yes'
            THEN `Monthly Charges`
            ELSE 0
        END
    ) AS revenue_lost,
    
    ROUND(
        SUM(
            CASE
                WHEN `Churn Label` = 'Yes'
                THEN 1
                ELSE 0
            END
        ) / COUNT(*) * 100,
        2
    ) AS churn_rate_pct

FROM telco_churn

GROUP BY
    `Contract`,
    `Internet Service`

ORDER BY revenue_lost DESC;
"""

cursor.execute(query3)

result = cursor.fetchall()

df_query3 = pd.DataFrame(
    result,
    columns=[
        "Contract",
        "Internet Service",
        "Total Customers",
        "Churned Customers",
        "Revenue Lost",
        "Churn Rate (%)"
    ]
)

df_query3

,Contract,Internet Service,Total Customers,Churned Customers,Revenue Lost,Churn Rate (%)
0,Month-to-month,Fiber optic,2128,1162,100481.999992,54.61
1,Month-to-month,DSL,1223,394,18367.250031,32.22
2,One year,Fiber optic,539,104,10571.950005,19.29
3,One year,DSL,570,53,3356.250002,9.30
4,Two year,Fiber optic,429,31,3246.099991,7.23
5,Month-to-month,No,524,99,1997.850000,18.89
6,Two year,DSL,628,12,805.699997,1.91
7,One year,No,364,9,190.249996,2.47
8,Two year,No,638,5,113.500000,0.78


### Observation

- The **Month-to-month + Fiber optic** segment has the largest churn-associated monthly revenue at approximately **100,482**.
- This segment contains **2,128 customers**, of whom **1,162 churned**, resulting in a churn rate of **54.61%**.
- The Month-to-month + DSL segment is the second-largest by churn-associated monthly revenue at approximately **18,367**.
- The results show why segment size and revenue exposure should be considered alongside churn rate.

## 4. Tenure Cohort Analysis
Bucket customers into named tenure cohorts (0-6mo, 7-12mo, 13-24mo, 25-48mo, 49mo+) 
using CASE WHEN, and compare churn rate and average revenue across cohorts.

In [6]:
query4 = """
SELECT
    CASE
        WHEN `Tenure Months` <= 6 THEN '0-6 months'
        WHEN `Tenure Months` <= 12 THEN '7-12 months'
        WHEN `Tenure Months` <= 24 THEN '13-24 months'
        WHEN `Tenure Months` <= 48 THEN '25-48 months'
        ELSE '49+ months'
    END AS tenure_cohort,

    COUNT(*) AS total_customers,

    SUM(
        CASE
            WHEN `Churn Label` = 'Yes'
            THEN 1
            ELSE 0
        END
    ) AS churned_customers,

    ROUND(
        SUM(
            CASE
                WHEN `Churn Label` = 'Yes'
                THEN 1
                ELSE 0
            END
        ) / COUNT(*) * 100,
        2
    ) AS churn_rate_pct,

    ROUND(
        AVG(`Monthly Charges`),
        2
    ) AS avg_monthly_revenue

FROM telco_churn

GROUP BY tenure_cohort

ORDER BY
    CASE tenure_cohort
        WHEN '0-6 months' THEN 1
        WHEN '7-12 months' THEN 2
        WHEN '13-24 months' THEN 3
        WHEN '25-48 months' THEN 4
        WHEN '49+ months' THEN 5
    END;
"""

cursor.execute(query4)

result = cursor.fetchall()

df_query4 = pd.DataFrame(
    result,
    columns=[
        "Tenure Cohort",
        "Total Customers",
        "Churned Customers",
        "Churn Rate (%)",
        "Avg Monthly Revenue"
    ]
)

df_query4

,Tenure Cohort,Total Customers,Churned Customers,Churn Rate (%),Avg Monthly Revenue
0,0-6 months,1481,784,52.94,54.74
1,7-12 months,705,253,35.89,58.95
2,13-24 months,1024,294,28.71,61.36
3,25-48 months,1594,325,20.39,65.93
4,49+ months,2239,213,9.51,73.95


### Observation

- Churn rate is highest among customers with **0–6 months of tenure at 52.94%**.
- Churn decreases progressively across the tenure cohorts, reaching **9.51%** among customers with 49+ months of tenure.
- Average monthly charges increase from **54.74** in the 0–6 month cohort to **73.95** in the 49+ month cohort.
- This shows a strong descriptive relationship between customer tenure and churn in the dataset.

## 5. Payment Method Within Contract Type
Check whether the "Electronic check churns more" pattern holds across all 
contract types, or if it's concentrated only within Month-to-month customers.

In [7]:
query5 = """
SELECT
    `Contract`,
    `Payment Method`,
    COUNT(*) AS total_customers,

    SUM(
        CASE
            WHEN `Churn Label` = 'Yes'
            THEN 1
            ELSE 0
        END
    ) AS churned_customers,

    ROUND(
        SUM(
            CASE
                WHEN `Churn Label` = 'Yes'
                THEN 1
                ELSE 0
            END
        ) / COUNT(*) * 100,
        2
    ) AS churn_rate_pct

FROM telco_churn

GROUP BY
    `Contract`,
    `Payment Method`

ORDER BY
    `Contract`,
    churn_rate_pct DESC;
"""

cursor.execute(query5)

result = cursor.fetchall()

df_query5 = pd.DataFrame(
    result,
    columns=[
        "Contract",
        "Payment Method",
        "Total Customers",
        "Churned Customers",
        "Churn Rate (%)"
    ]
)

df_query5

,Contract,Payment Method,Total Customers,Churned Customers,Churn Rate (%)
0,Month-to-month,Electronic check,1850,994,53.73
1,Month-to-month,Bank transfer (automatic),589,201,34.13
2,Month-to-month,Credit card (automatic),543,178,32.78
3,Month-to-month,Mailed check,893,282,31.58
4,One year,Electronic check,347,64,18.44
5,One year,Credit card (automatic),398,41,10.30
6,One year,Bank transfer (automatic),391,38,9.72
7,One year,Mailed check,337,23,6.82
8,Two year,Electronic check,168,13,7.74
9,Two year,Bank transfer (automatic),564,19,3.37


### Observation

- Electronic check has the highest churn rate within each contract type in this dataset.
- The highest observed rate is **53.73%** among month-to-month customers using electronic check.
- Churn rates are substantially lower for one-year and two-year contracts across payment methods.
- The results suggest that payment method and contract type should be examined together rather than independently when analyzing churn patterns.

## 6. Add-on Service Bundling Effect
Compare churn rates across customers with 0, 1, 2, 3, and 4 additional services.


In [10]:
query6 = """
SELECT
    addon_count,
    COUNT(*) AS total_customers,

    SUM(
        CASE
            WHEN `Churn Label` = 'Yes'
            THEN 1
            ELSE 0
        END
    ) AS churned_customers,

    ROUND(
        SUM(
            CASE
                WHEN `Churn Label` = 'Yes'
                THEN 1
                ELSE 0
            END
        ) / COUNT(*) * 100,
        2
    ) AS churn_rate_pct

FROM (
    SELECT
        `Churn Label`,

        (CASE WHEN `Online Security` = 'Yes' THEN 1 ELSE 0 END)
      + (CASE WHEN `Online Backup` = 'Yes' THEN 1 ELSE 0 END)
      + (CASE WHEN `Device Protection` = 'Yes' THEN 1 ELSE 0 END)
      + (CASE WHEN `Tech Support` = 'Yes' THEN 1 ELSE 0 END)
        AS addon_count

    FROM telco_churn
) AS sub

GROUP BY addon_count

ORDER BY addon_count;
"""

cursor.execute(query6)

result = cursor.fetchall()

df_query6 = pd.DataFrame(
    result,
    columns=[
        "Add-on Count",
        "Total Customers",
        "Churned Customers",
        "Churn Rate (%)"
    ]
)

df_query6

,Add-on Count,Total Customers,Churned Customers,Churn Rate (%)
0,0,2793,831,29.75
1,1,1467,570,38.85
2,2,1372,326,23.76
3,3,941,117,12.43
4,4,470,25,5.32


### Observation

- Customers with exactly **1 add-on service have the highest churn rate at 38.85%**.
- Churn decreases substantially among customers with 2 or more add-on services.
- Customers with all 4 analyzed add-on services have a churn rate of only **5.32%**.
- The relationship is not simply "having any add-on reduces churn," since the 1-add-on group has a higher churn rate than the 0-add-on group.
- The results indicate that the **number of services bundled** is more informative than a simple yes/no distinction for add-on adoption.

## 7. Ranking Segments with Window Functions
Use RANK() to produce a ranked list of all Contract + Payment Method combinations, 
ordered from highest to lowest churn rate.
markdown

In [11]:
query7 = """
SELECT
    `Contract`,
    `Payment Method`,
    total_customers,
    churned_customers,
    churn_rate_pct,

    RANK() OVER (
        ORDER BY churn_rate_pct DESC
    ) AS churn_rank

FROM (
    SELECT
        `Contract`,
        `Payment Method`,

        COUNT(*) AS total_customers,

        SUM(
            CASE
                WHEN `Churn Label` = 'Yes'
                THEN 1
                ELSE 0
            END
        ) AS churned_customers,

        ROUND(
            SUM(
                CASE
                    WHEN `Churn Label` = 'Yes'
                    THEN 1
                    ELSE 0
                END
            ) / COUNT(*) * 100,
            2
        ) AS churn_rate_pct

    FROM telco_churn

    GROUP BY
        `Contract`,
        `Payment Method`

) AS segment_summary

ORDER BY churn_rank;
"""

cursor.execute(query7)

result = cursor.fetchall()

df_query7 = pd.DataFrame(
    result,
    columns=[
        "Contract",
        "Payment Method",
        "Total Customers",
        "Churned Customers",
        "Churn Rate (%)",
        "Churn Rank"
    ]
)

df_query7

,Contract,Payment Method,Total Customers,Churned Customers,Churn Rate (%),Churn Rank
0,Month-to-month,Electronic check,1850,994,53.73,1
1,Month-to-month,Bank transfer (automatic),589,201,34.13,2
2,Month-to-month,Credit card (automatic),543,178,32.78,3
3,Month-to-month,Mailed check,893,282,31.58,4
4,One year,Electronic check,347,64,18.44,5
5,One year,Credit card (automatic),398,41,10.30,6
6,One year,Bank transfer (automatic),391,38,9.72,7
7,Two year,Electronic check,168,13,7.74,8
8,One year,Mailed check,337,23,6.82,9
9,Two year,Bank transfer (automatic),564,19,3.37,10


### Observation

- The highest-ranked segment is **Month-to-month + Electronic check**, with a churn rate of **53.73%**.
- The next highest-ranked segments are also month-to-month customers using other payment methods.
- Two-year contract segments occupy the lower end of the ranking.
- The ranking demonstrates how window functions can be used to compare multiple customer segments based on a calculated metric.

## 8. New Customer Revenue Exposure

Calculate the total monthly revenue currently associated with active customers
with tenure ≤ 3 months and estimate the revenue exposure using the historical
churn rate of customers in the same tenure group.


In [18]:
query8 = """
SELECT
    SUM(
        CASE
            WHEN `Churn Label` = 'No'
            THEN `Monthly Charges`
            ELSE 0
        END
    ) AS active_revenue_in_risk_window,

    ROUND(
        SUM(
            CASE
                WHEN `Churn Label` = 'No'
                THEN `Monthly Charges`
                ELSE 0
            END
        )
        *
        (
            SUM(
                CASE
                    WHEN `Churn Label` = 'Yes'
                    THEN 1
                    ELSE 0
                END
            ) / COUNT(*)
        ),
        2
    ) AS estimated_revenue_exposure

FROM telco_churn

WHERE `Tenure Months` <= 3;
"""

cursor.execute(query8)

result = cursor.fetchall()

df_query8 = pd.DataFrame(
    result,
    columns=[
        "Active Revenue in Risk Window",
        "Estimated Revenue Exposure"
    ]
)

df_query8

,Active Revenue in Risk Window,Estimated Revenue Exposure
0,20192.799967,11351.32


### Observation

- The query identifies the monthly revenue associated with active customers who have been with the company for 3 months or less.
- The estimated revenue exposure is calculated using the historical churn rate of the same ≤3-month tenure group.
- This provides a cohort-specific estimate rather than applying the overall customer churn rate.

## 9. Compound Risk Segment
Build a specific high-risk segment (Senior Citizen + No Dependents + Month-to-month 
contract) and compare its churn rate to the overall baseline, to test whether 
stacking risk factors compounds churn risk.

In [14]:
query9 = """
SELECT
    COUNT(*) AS segment_size,

    SUM(
        CASE
            WHEN `Churn Label` = 'Yes'
            THEN 1
            ELSE 0
        END
    ) AS churned_customers,

    ROUND(
        SUM(
            CASE
                WHEN `Churn Label` = 'Yes'
                THEN 1
                ELSE 0
            END
        ) / COUNT(*) * 100,
        2
    ) AS churn_rate_pct

FROM telco_churn

WHERE `Senior Citizen` = 'Yes'
  AND `Dependents` = 'No'
  AND `Contract` = 'Month-to-month';
"""

cursor.execute(query9)

result = cursor.fetchall()

df_query9 = pd.DataFrame(
    result,
    columns=[
        "Segment Size",
        "Churned Customers",
        "Churn Rate (%)"
    ]
)

df_query9

,Segment Size,Churned Customers,Churn Rate (%)
0,764,429,56.15


### Observation

- The selected segment contains **764 customers**, of whom **429 churned**.
- The segment has a churn rate of **56.15%**, which is substantially higher than the overall dataset churn rate of **26.54%**.
- This segment combines three customer characteristics: Senior Citizen, No Dependents, and Month-to-month contract.
- The result identifies a high-churn customer segment for further investigation, but it does not establish that these factors independently or jointly cause churn.

## 10. Gender × Senior Citizen — Compound Demographic Check
Check whether Senior Citizen status affects churn consistently across both genders, 
or whether there's a hidden interaction between Gender and age that wasn't visible 
when looking at each factor separately in EDA.

In [15]:
query10 = """
SELECT
    `Gender`,
    `Senior Citizen`,
    COUNT(*) AS total_customers,

    SUM(
        CASE
            WHEN `Churn Label` = 'Yes'
            THEN 1
            ELSE 0
        END
    ) AS churned_customers,

    ROUND(
        SUM(
            CASE
                WHEN `Churn Label` = 'Yes'
                THEN 1
                ELSE 0
            END
        ) / COUNT(*) * 100,
        2
    ) AS churn_rate_pct

FROM telco_churn

GROUP BY
    `Gender`,
    `Senior Citizen`

ORDER BY
    `Gender`,
    `Senior Citizen`;
"""

cursor.execute(query10)

result = cursor.fetchall()

df_query10 = pd.DataFrame(
    result,
    columns=[
        "Gender",
        "Senior Citizen",
        "Total Customers",
        "Churned Customers",
        "Churn Rate (%)"
    ]
)

df_query10

,Gender,Senior Citizen,Total Customers,Churned Customers,Churn Rate (%)
0,Female,No,2920,699,23.94
1,Female,Yes,568,240,42.25
2,Male,No,2981,694,23.28
3,Male,Yes,574,236,41.11


### Observation

- Churn rates are similar for non-senior females (**23.94%**) and non-senior males (**23.28%**).
- Among senior citizens, churn is higher for both females (**42.25%**) and males (**41.11%**).
- The difference between genders is relatively small within both senior-citizen groups.
- In this descriptive analysis, Senior Citizen status shows a much larger difference in churn rate than Gender.

## Summary — Key SQL Findings

1. **Revenue at risk**: Churned customers account for **30.50%** of total monthly
   charges (**$139,130.85 of $456,116.60**).

2. **Highest-risk segment by revenue**: **Month-to-month + Fiber optic** customers
   had the largest churn-associated monthly revenue (**~$100,482.00**), with a
   **54.61%** churn rate.

3. **Tenure cohorts**: Churn drops from **52.94% (0–6 months)** to **9.51%
   (49+ months)** — highlighting a strong early-tenure risk window.

4. **Payment method varies by contract**: Electronic check has the highest churn
   within every contract type, peaking at **53.73%** for **Month-to-month +
   Electronic check**.

5. **Add-on bundling is non-linear**: Customers with exactly 1 add-on churn at
   **38.85%**, which is higher than customers with 0 add-ons (**29.75%**).
   Churn then drops substantially for customers with 2 or more add-ons.

6. **Compound risk segment**: **Senior Citizen + No Dependents +
   Month-to-month** customers have a **56.15%** churn rate, compared with the
   overall baseline of **26.54%**. This identifies a particularly high-churn
   combination of customer characteristics.

7. **High-value customers at risk**: **1,061 active customers** have both
   above-average CLTV and above-average IBM Churn Score, providing a
   descriptive high-value/high-risk customer group for retention analysis.

These SQL findings extend the EDA results with revenue exposure, segment
interactions, tenure patterns, payment and contract combinations, service
bundling, and compound risk profiling. Together, they provide additional
business context for feature selection, customer segmentation, and the final
machine learning and retention analysis.